# 03 - Clustering Avancado (Etapa 3)

Este notebook inicia e consolida a implementacao da etapa 3 de agrupamento:

- Aplicar um segundo algoritmo de clustering (DBSCAN)
- Comparar com o baseline K-Means
- Avaliar com pelo menos duas metricas
- Analisar parametros utilizados
- Interpretar e nomear perfis de cluster
- Gerar estrutura final para alimentar a etapa de Inteligencia Computacional

## 1. Set Up Environment and Dependencies

Importacao de bibliotecas, configuracoes globais e verificacao de versoes para reprodutibilidade.

In [ ]:
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid")

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

## 2. Define Configuration and Constants

Bloco central de paths, parametros de busca e colunas esperadas.

In [ ]:
@dataclass(frozen=True)
class ClusteringConfig:
    base_path: Path = Path("../data/processed/telco_clustering_base.csv")
    clean_path: Path = Path("../data/processed/telco_clean.csv")
    with_clusters_path: Path = Path("../data/processed/telco_with_clusters.csv")
    scaler_path: Path = Path("../models/scaler.joblib")
    kmeans_path: Path = Path("../models/kmeans_model.joblib")
    comparison_out_path: Path = Path("../reports/comparison_summary.csv")
    figure_dir: Path = Path("../reports/figures")
    eps_grid: Tuple[float, ...] = (0.30, 0.40, 0.50, 0.60, 0.80, 1.00, 1.20)
    min_samples_grid: Tuple[int, ...] = (3, 5, 10, 15)


CFG = ClusteringConfig()

RAW_NUM_COLS = ["tenure", "MonthlyCharges"]
RAW_CAT_COLS = [
    "Contract",
    "InternetService",
    "TechSupport",
    "OnlineSecurity",
    "PaymentMethod",
]

INTERNET_LEVELS = ["DSL", "Fiber optic", "No"]
PAYMENT_LEVELS = [
    "Bank transfer (automatic)",
    "Credit card (automatic)",
    "Electronic check",
    "Mailed check",
]

FEATURE_COLS = [
    "tenure_scaled",
    "MonthlyCharges_scaled",
    "Contract_ordinal",
    "TechSupport_enc",
    "OnlineSecurity_enc",
    "InternetService_DSL",
    "InternetService_Fiber optic",
    "InternetService_No",
    "PaymentMethod_Bank transfer (automatic)",
    "PaymentMethod_Credit card (automatic)",
    "PaymentMethod_Electronic check",
    "PaymentMethod_Mailed check",
]

print(CFG)

## 3. Implement Core Data Structures

Estruturas para armazenar metricas por algoritmo e resultados da calibracao.

In [ ]:
@dataclass
class ClusterMetrics:
    algorithm: str
    n_clusters: int
    noise_pct: float
    silhouette: float
    davies_bouldin: float
    calinski_harabasz: float


@dataclass
class DBSCANTrial:
    eps: float
    min_samples: int
    n_clusters: int
    noise_pct: float
    silhouette: float
    davies_bouldin: float

## 4. Implement Main Processing Functions

Funcoes modulares para preprocessamento, avaliacao, busca de parametros, comparacao e exportacao.

In [ ]:
def validate_input_dataframe(df: pd.DataFrame, required_cols: List[str]) -> None:
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Colunas obrigatorias ausentes: {missing}")
    if df.empty:
        raise ValueError("DataFrame de entrada vazio.")


def _safe_internal_metrics(X: np.ndarray, labels: np.ndarray) -> Tuple[float, float, float]:
    unique_labels = np.unique(labels)
    n_clusters = len(unique_labels)
    if n_clusters <= 1:
        return (np.nan, np.nan, np.nan)
    sil = silhouette_score(X, labels)
    dbi = davies_bouldin_score(X, labels)
    ch = calinski_harabasz_score(X, labels)
    return (float(sil), float(dbi), float(ch))


def _safe_internal_metrics_ignore_noise(X: np.ndarray, labels: np.ndarray) -> Tuple[float, float, float]:
    mask = labels != -1
    if mask.sum() < 3:
        return (np.nan, np.nan, np.nan)
    labels_no_noise = labels[mask]
    X_no_noise = X[mask]
    unique_labels = np.unique(labels_no_noise)
    if len(unique_labels) <= 1:
        return (np.nan, np.nan, np.nan)
    sil = silhouette_score(X_no_noise, labels_no_noise)
    dbi = davies_bouldin_score(X_no_noise, labels_no_noise)
    ch = calinski_harabasz_score(X_no_noise, labels_no_noise)
    return (float(sil), float(dbi), float(ch))


def build_clustering_matrix(base_df: pd.DataFrame, scaler) -> Tuple[pd.DataFrame, np.ndarray]:
    work = base_df.copy()

    contract_map = {"Month-to-month": 0, "One year": 1, "Two year": 2}
    binary_map = {"Yes": 1, "No": 0, "No internet service": -1}

    work["Contract_ordinal"] = work["Contract"].map(contract_map).fillna(0).astype(int)
    work["TechSupport_enc"] = work["TechSupport"].map(binary_map).fillna(0).astype(int)
    work["OnlineSecurity_enc"] = work["OnlineSecurity"].map(binary_map).fillna(0).astype(int)

    internet_dummies = pd.get_dummies(work["InternetService"], prefix="InternetService")
    payment_dummies = pd.get_dummies(work["PaymentMethod"], prefix="PaymentMethod")

    for level in INTERNET_LEVELS:
        col = f"InternetService_{level}"
        if col not in internet_dummies.columns:
            internet_dummies[col] = 0

    for level in PAYMENT_LEVELS:
        col = f"PaymentMethod_{level}"
        if col not in payment_dummies.columns:
            payment_dummies[col] = 0

    work[["tenure_scaled", "MonthlyCharges_scaled"]] = scaler.transform(work[RAW_NUM_COLS])

    matrix_df = pd.concat(
        [
            work,
            internet_dummies,
            payment_dummies,
        ],
        axis=1,
    )

    for c in FEATURE_COLS:
        if c not in matrix_df.columns:
            matrix_df[c] = 0

    X = matrix_df[FEATURE_COLS].astype(float).to_numpy()
    return matrix_df, X


def evaluate_kmeans_baseline(X: np.ndarray, kmeans_model) -> Tuple[np.ndarray, ClusterMetrics]:
    labels = kmeans_model.predict(X)
    sil, dbi, ch = _safe_internal_metrics(X, labels)
    metrics = ClusterMetrics(
        algorithm="kmeans",
        n_clusters=int(len(np.unique(labels))),
        noise_pct=0.0,
        silhouette=sil,
        davies_bouldin=dbi,
        calinski_harabasz=ch,
    )
    return labels, metrics


def dbscan_grid_search(X: np.ndarray, eps_grid: Tuple[float, ...], min_samples_grid: Tuple[int, ...]) -> pd.DataFrame:
    trials: List[DBSCANTrial] = []
    for eps in eps_grid:
        for min_samples in min_samples_grid:
            model = DBSCAN(eps=eps, min_samples=min_samples)
            labels = model.fit_predict(X)

            unique_non_noise = sorted(set(labels) - {-1})
            n_clusters = len(unique_non_noise)
            noise_pct = float((labels == -1).mean() * 100)
            sil, dbi, _ = _safe_internal_metrics_ignore_noise(X, labels)

            trials.append(
                DBSCANTrial(
                    eps=float(eps),
                    min_samples=int(min_samples),
                    n_clusters=int(n_clusters),
                    noise_pct=noise_pct,
                    silhouette=sil,
                    davies_bouldin=dbi,
                )
            )

    trial_df = pd.DataFrame([t.__dict__ for t in trials])
    return trial_df.sort_values(["silhouette", "davies_bouldin"], ascending=[False, True], na_position="last")


def choose_dbscan_params(trial_df: pd.DataFrame) -> Tuple[float, int]:
    valid = trial_df.dropna(subset=["silhouette", "davies_bouldin"]).copy()
    if valid.empty:
        raise ValueError("DBSCAN nao encontrou combinacoes validas com 2+ clusters sem ruido total.")
    best = valid.sort_values(["silhouette", "davies_bouldin"], ascending=[False, True]).iloc[0]
    return float(best["eps"]), int(best["min_samples"])


def train_dbscan_final(X: np.ndarray, eps: float, min_samples: int) -> Tuple[np.ndarray, ClusterMetrics]:
    model = DBSCAN(eps=eps, min_samples=min_samples)
    labels = model.fit_predict(X)
    sil, dbi, ch = _safe_internal_metrics_ignore_noise(X, labels)

    n_clusters = int(len(set(labels) - {-1}))
    noise_pct = float((labels == -1).mean() * 100)

    metrics = ClusterMetrics(
        algorithm="dbscan",
        n_clusters=n_clusters,
        noise_pct=noise_pct,
        silhouette=sil,
        davies_bouldin=dbi,
        calinski_harabasz=ch,
    )
    return labels, metrics


def pick_winner(km: ClusterMetrics, db: ClusterMetrics) -> str:
    # Criterio principal: maior silhouette; desempate: menor Davies-Bouldin.
    if np.isnan(db.silhouette) or np.isnan(db.davies_bouldin):
        return "kmeans"
    if db.silhouette > km.silhouette:
        return "dbscan"
    if np.isclose(db.silhouette, km.silhouette) and db.davies_bouldin < km.davies_bouldin:
        return "dbscan"
    return "kmeans"


def build_cluster_summary(df: pd.DataFrame, labels: np.ndarray, algorithm_name: str) -> pd.DataFrame:
    temp = df.copy()
    temp["cluster"] = labels

    summary = (
        temp.groupby("cluster", dropna=False)
        .agg(
            tamanho=("customerID", "count"),
            tenure_medio=("tenure", "mean"),
            monthly_medio=("MonthlyCharges", "mean"),
            churn_pct=("Churn_label", lambda s: (s == "Yes").mean() * 100),
            contrato_dominante=("Contract", lambda s: s.mode().iloc[0] if not s.mode().empty else "N/A"),
            internet_dominante=("InternetService", lambda s: s.mode().iloc[0] if not s.mode().empty else "N/A"),
        )
        .reset_index()
    )
    summary["algoritmo"] = algorithm_name
    summary["cluster_label"] = summary.apply(_semantic_label_rule, axis=1)
    return summary


def _semantic_label_rule(row: pd.Series) -> str:
    if int(row["cluster"]) == -1:
        return "Atipicos (ruido DBSCAN)"

    churn = float(row["churn_pct"])
    mensalidade = float(row["monthly_medio"])
    contrato = str(row["contrato_dominante"])
    internet = str(row["internet_dominante"])

    if churn >= 25 and contrato == "Month-to-month" and mensalidade >= 65:
        return "Insatisfeito com servicos"
    if churn <= 12 and (contrato == "Two year" or internet == "No"):
        return "Economico estavel"
    if churn >= 20:
        return "Risco moderado"
    return "Perfil intermediario"


def overwrite_output_with_winner(source_df: pd.DataFrame, labels: np.ndarray, labels_map: Dict[int, str], algorithm: str, out_path: Path) -> pd.DataFrame:
    out_df = source_df.copy()
    out_df["cluster"] = labels
    out_df["cluster_label"] = pd.Series(labels).map(labels_map).fillna("Perfil nao definido").values
    out_df["clustering_algorithm"] = algorithm
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(out_path, index=False)
    return out_df


def plot_dbscan_search(trial_df: pd.DataFrame, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    pivot = trial_df.pivot(index="min_samples", columns="eps", values="silhouette")
    plt.figure(figsize=(10, 4))
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGnBu")
    plt.title("DBSCAN - Silhouette por eps e min_samples")
    plt.tight_layout()
    plt.savefig(out_dir / "10_dbscan_param_search.png", dpi=150)
    plt.show()


def plot_algorithm_comparison(X: np.ndarray, kmeans_labels: np.ndarray, dbscan_labels: np.ndarray, comparison_df: pd.DataFrame, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    pca = PCA(n_components=2, random_state=42)
    X2 = pca.fit_transform(X)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].scatter(X2[:, 0], X2[:, 1], c=kmeans_labels, s=8, alpha=0.6, cmap="viridis")
    axes[0].set_title("K-Means (PCA)")

    axes[1].scatter(X2[:, 0], X2[:, 1], c=dbscan_labels, s=8, alpha=0.6, cmap="tab10")
    axes[1].set_title("DBSCAN (PCA)")

    for ax in axes:
        ax.set_xlabel("PCA 1")
        ax.set_ylabel("PCA 2")

    plt.tight_layout()
    plt.savefig(out_dir / "12_pca_comparison_2algorithms.png", dpi=150)
    plt.show()

    plt.figure(figsize=(8, 4))
    melted = comparison_df.melt(
        id_vars="algorithm",
        value_vars=["silhouette", "davies_bouldin"],
        var_name="metrica",
        value_name="valor",
    )
    sns.barplot(data=melted, x="metrica", y="valor", hue="algorithm")
    plt.title("Comparacao de metricas internas")
    plt.tight_layout()
    plt.savefig(out_dir / "14_metrics_comparison.png", dpi=150)
    plt.show()

## 5. Add Input Validation and Error Handling

Checagens para garantir existencia de arquivos e schema minimo antes da execucao do pipeline.

In [ ]:
for path in [CFG.base_path, CFG.with_clusters_path, CFG.scaler_path, CFG.kmeans_path]:
    if not path.exists():
        raise FileNotFoundError(f"Arquivo obrigatorio nao encontrado: {path}")

base_df = pd.read_csv(CFG.base_path)
validate_input_dataframe(base_df, ["customerID", "Churn_label", *RAW_NUM_COLS, *RAW_CAT_COLS])

with_clusters_df = pd.read_csv(CFG.with_clusters_path)
validate_input_dataframe(with_clusters_df, ["customerID"])

print("Validacoes basicas concluidas.")
print("base_df:", base_df.shape, "with_clusters_df:", with_clusters_df.shape)

## 6. Create a Minimal Execution Pipeline

Pipeline completo: baseline K-Means, busca e treino DBSCAN, comparacao, escolha do vencedor e geracao da saida final.

In [ ]:
scaler = joblib.load(CFG.scaler_path)
kmeans_model = joblib.load(CFG.kmeans_path)

matrix_df, X = build_clustering_matrix(base_df, scaler)

kmeans_labels, kmeans_metrics = evaluate_kmeans_baseline(X, kmeans_model)
print("K-Means metricas:", kmeans_metrics)

dbscan_trials = dbscan_grid_search(X, CFG.eps_grid, CFG.min_samples_grid)
best_eps, best_min_samples = choose_dbscan_params(dbscan_trials)
print(f"DBSCAN parametros escolhidos -> eps={best_eps}, min_samples={best_min_samples}")

dbscan_labels, dbscan_metrics = train_dbscan_final(X, best_eps, best_min_samples)
print("DBSCAN metricas:", dbscan_metrics)

winner = pick_winner(kmeans_metrics, dbscan_metrics)
print("Algoritmo vencedor:", winner)

comparison_df = pd.DataFrame(
    [
        kmeans_metrics.__dict__,
        dbscan_metrics.__dict__,
    ]
)
comparison_df = comparison_df[[
    "algorithm",
    "n_clusters",
    "noise_pct",
    "silhouette",
    "davies_bouldin",
    "calinski_harabasz",
]]
comparison_df.to_csv(CFG.comparison_out_path, index=False)

if winner == "dbscan":
    final_labels = dbscan_labels
    final_algo = "dbscan"
else:
    final_labels = kmeans_labels
    final_algo = "kmeans"

cluster_summary = build_cluster_summary(base_df, final_labels, final_algo)
cluster_map = cluster_summary.set_index("cluster")["cluster_label"].to_dict()

# Reordena labels para acompanhar a base de saida pelo customerID.
label_df = base_df[["customerID"]].copy()
label_df["cluster"] = final_labels

output_base = with_clusters_df.drop(columns=["cluster", "cluster_label", "clustering_algorithm"], errors="ignore")
output_joined = output_base.merge(label_df, on="customerID", how="left")

final_output_df = overwrite_output_with_winner(
    source_df=output_joined,
    labels=output_joined["cluster"].to_numpy(),
    labels_map=cluster_map,
    algorithm=final_algo,
    out_path=CFG.with_clusters_path,
)

plot_dbscan_search(dbscan_trials, CFG.figure_dir)
plot_algorithm_comparison(X, kmeans_labels, dbscan_labels, comparison_df, CFG.figure_dir)

cluster_summary.head()

## 7. Write Unit Tests for Core Logic

Testes simples para garantir comportamento esperado em funcoes criticas.

In [ ]:
def test_safe_metrics_single_cluster_returns_nan() -> None:
    X_t = np.array([[0.0, 0.0], [1.0, 1.0], [1.1, 1.0]])
    labels_t = np.array([0, 0, 0])
    sil, dbi, ch = _safe_internal_metrics(X_t, labels_t)
    assert np.isnan(sil)
    assert np.isnan(dbi)
    assert np.isnan(ch)


def test_pick_winner_prefers_better_silhouette() -> None:
    km = ClusterMetrics("kmeans", 2, 0.0, 0.30, 1.20, 100.0)
    db = ClusterMetrics("dbscan", 3, 5.0, 0.35, 1.10, 90.0)
    assert pick_winner(km, db) == "dbscan"


def test_pick_winner_fallback_when_dbscan_invalid() -> None:
    km = ClusterMetrics("kmeans", 2, 0.0, 0.30, 1.20, 100.0)
    db = ClusterMetrics("dbscan", 0, 100.0, np.nan, np.nan, np.nan)
    assert pick_winner(km, db) == "kmeans"


def run_unit_tests() -> None:
    tests = [
        test_safe_metrics_single_cluster_returns_nan,
        test_pick_winner_prefers_better_silhouette,
        test_pick_winner_fallback_when_dbscan_invalid,
    ]
    for t in tests:
        t()
    print(f"{len(tests)} testes executados com sucesso.")


run_unit_tests()

## 8. Run and Inspect Output

Inspecao final dos artefatos gerados para confirmar consolidacao da etapa de agrupamento.

In [ ]:
print("\nResumo comparativo:")
display(comparison_df)

print("\nTop 10 combinacoes DBSCAN:")
display(dbscan_trials.head(10))

print("\nPerfis finais (resumo):")
display(cluster_summary)

print("\nAmostra da saida final para IC:")
display(final_output_df[["customerID", "cluster", "cluster_label", "clustering_algorithm"]].head())

print("\nArquivos gerados:")
print("-", CFG.comparison_out_path)
print("-", CFG.with_clusters_path)
print("-", CFG.figure_dir / "10_dbscan_param_search.png")
print("-", CFG.figure_dir / "12_pca_comparison_2algorithms.png")
print("-", CFG.figure_dir / "14_metrics_comparison.png")